**Nothing should be different for a live simulation vs a loaded simulation**

In [ ]:
import SAI_functions_20260207 as SAIfunc, SAI_svg, simclass as mem
from numpy.strings import mod as npformat
import numpy as np
import gradio as gr
import json

#### FOR RUNNING

**RUN THIS ONLY WHEN NEW INITIAL STATE DESIRED**

In [2]:
nV = 5
p0 = 0.5
pE = 2/(nV-1)
a0 = a1 = 0.4

sim_params = {
    # KEYS NOT TO BE CHANGED FOR ANY REASON
    'nV': nV,
    'p0': p0,
    'pE': pE,
    'a0': a0,
    'a1': a0
}
Vnought, Enought = SAIfunc.init_state(nV, p0, pE)

In [ ]:
Vclasses = np.where(Vnought, '_1', '_0')
Eclasses = np.where(Enought, 'on', 'of')

In [18]:
trialsim = mem.simmem(20, 10, nV, Vnought, Enought)

#### EVENT HANDLERS

In [4]:
def prev_clicked(sliderval):
    new_val = (sliderval - 1) if (sliderval > 0) else (sliderval)
    return gr.update(maximum=len(trialsim)-1, value=new_val)

In [5]:
def next_clicked(sliderval):
    new_val = (sliderval + 1) if (sliderval < len(trialsim)-1) else (sliderval)
    return gr.update(maximum=len(trialsim)-1, value=new_val)

In [6]:
def compute_clicked():
    trialsim.push( *SAIfunc.next_state(nV, *trialsim[-1], [a0, a1]) )
    new_max = len(trialsim)-1
    return gr.update(maximum=new_max, value=new_max)


In [ ]:
def slider_changed(sliderval):
    # old_sliderval = trialsim.displayed
    # Vold, Eold = trialsim[old_sliderval]
    Vnew, Enew = trialsim[sliderval]
    trialsim.displayed = sliderval
    # return SAI_svg.style_updates_as_json(
    #     Vold, Eold, Vnew, Enew
    # )
    return SAI_svg.all_styles_as_json()

In [ ]:
gr.close_all()
trialsim = mem.simmem(20, 10, nV, Vnought, Enought)
trialsim.displayed = 0 # keep track of which is rendered, to compute diffs
with gr.Blocks(analytics_enabled=False) as UI:
    with gr.Row():
        with gr.Column(min_width=750):
            frame = gr.HTML(SAI_svg.svg_setup(300, nV, Vclasses, Eclasses))
            style_data = gr.TextArea(value='---', visible='hidden')
        with gr.Column():
            with gr.Row():
                prev_btn = gr.Button('Prev', min_width=60)
                next_btn = gr.Button('Next', min_width=60)
            frame_sldr = gr.Slider(minimum=0, maximum=len(trialsim)-1, value=0, step=1, min_width=300, elem_id='SAI_slider')
            with gr.Row():
                compute_btn = gr.Button('Compute next and jump to', scale=2, min_width=250)
                compute10_btn = gr.Button('Compute 10 next', scale=1, min_width=140)

    # All EVent Handlers here:
    prev_btn.click(prev_clicked, inputs=frame_sldr, outputs=frame_sldr)
    next_btn.click(next_clicked, inputs=frame_sldr, outputs=frame_sldr)
    compute_btn.click(compute_clicked, outputs=frame_sldr)
    "compute10_btn.click(compute_clicked, outputs=frame_sldr)"

    frame_sldr.change(slider_changed, inputs=frame_sldr, outputs=style_data)
    style_data.change(lambda anything:anything, inputs=style_data, js='(data) => update_graph(data)')

with open('update_graph.js', 'rt') as js:
    script = js.read()
    app, localurl, shareurl = UI.launch(js=script)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/pranav_ac/pyenv/SAI/lib/python3.13/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Users/pranav_ac/pyenv/SAI/lib/python3.13/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/Users/pranav_ac/pyenv/SAI/lib/python3.13/site-packages/gradio/blocks.py", line 2152, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/Users/pranav_ac/pyenv/SAI/lib/python3.13/site-packages/gradio/blocks.py", line 1629, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        f